<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-1 · Part 4: Single- and Multi-Objective Optimization

**Discomfort and energy describe the same cooling decision, but the objective determines how those outputs are compared.**

Part 3 changed the decision domain. This part returns to the continuous classroom domain and keeps the simulator and requirements fixed. Only the comparison rule changes.

| Kept fixed | Changed here | Resulting classification |
|:---|:---|:---|
| $x$, $\mathcal X$, $y=\operatorname{Sim}(x)$, constraints | Scalar $f$ or vector $\boldsymbol f$ | Single- or multi-objective optimization |


### 1 · Performance outputs are not yet a comparison rule

The simulator produces two raw performance outputs: discomfort \(D(u)\) and energy \(E(u)\). They describe a candidate. An objective states what “better” means.

A **single-objective formulation** supplies one scalar value:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad f(y;\lambda_E)=J(u;\lambda_E)=D(u)+\lambda_EE(u).$

Although this score combines two performance outputs, it is one objective because each candidate receives one scalar comparison value. Maximizing a benefit \(p(y)\) can use the same convention by minimizing \(-p(y)\).

A **multi-objective formulation** keeps the two objectives separate:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad \boldsymbol f(y)=\begin{bmatrix}f_D(y)\\f_E(y)\end{bmatrix}=\begin{bmatrix}D(u)\\E(u)\end{bmatrix}.$

There is no scalar score yet. The formulation first exposes which candidates trade discomfort against energy.


### 2 · Pareto dominance removes clearly inferior candidates

Candidate A **dominates** candidate B when A is no worse in every minimized objective and strictly better in at least one. A candidate that is not dominated belongs to the **Pareto front**.

The next figure plots feasible 0.5-grid candidates in the raw performance space \((E,D)\). Lower and farther left are better. The connected points are nondominated grid candidates.


In [ ]:
import numpy as np


TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Decision domain and requirement limits
MIN_COOLING = 0.0
MAX_COOLING = 5.0
MIN_TEMPERATURE = 20.0
MAX_TEMPERATURE = 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    """Expand x=[early cooling, late cooling] into the 12-step schedule."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[
        np.full(6, early_cooling),
        np.full(6, late_cooling),
    ]


def simulation_model(x):
    """Return y=Sim(x): the state path and raw performance outputs."""
    cooling_schedule = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]

    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )

    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule**2)
    return {
        "temperatures": temperatures,
        "discomfort": float(discomfort),
        "energy": float(energy),
    }


def objective_function(y, energy_weight=1.0):
    """Return f(y; lambda_E)=D(u)+lambda_E E(u)."""
    return y["discomfort"] + float(energy_weight) * y["energy"]


def inequality_constraints(x, y, energy_limit=MAX_ENERGY):
    """Return residuals in the feasible form g_j(x,y) <= 0."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    temperatures = y["temperatures"][1:]
    return {
        "early lower": MIN_COOLING - early_cooling,
        "early upper": early_cooling - MAX_COOLING,
        "late lower": MIN_COOLING - late_cooling,
        "late upper": late_cooling - MAX_COOLING,
        "temperature lower": MIN_TEMPERATURE - temperatures.min(),
        "temperature upper": temperatures.max() - MAX_TEMPERATURE,
        "energy": y["energy"] - float(energy_limit),
    }


def equality_constraints(x, y):
    """Return state-equation residuals h_t(x,y), which should equal zero."""
    cooling_schedule = expand_decision(x)
    temperatures = y["temperatures"]
    residuals = []
    for step, (outdoor, people, cooling) in enumerate(
        zip(OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule)
    ):
        predicted_next = (
            temperatures[step]
            + WEATHER_EXCHANGE * (outdoor - temperatures[step])
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
        residuals.append(temperatures[step + 1] - predicted_next)
    return np.asarray(residuals)


def evaluate_formulation(x, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate x, y, f, g, and h for one candidate decision."""
    x = tuple(map(float, x))
    y = simulation_model(x)
    g = inequality_constraints(x, y, energy_limit)
    h = equality_constraints(x, y)
    feasible = all(value <= 1e-10 for value in g.values()) and np.allclose(h, 0.0)
    return {
        "x": x,
        "y": y,
        "f": objective_function(y, energy_weight),
        "g": g,
        "h": h,
        "energy_weight": float(energy_weight),
        "energy_limit": float(energy_limit),
        "feasible": bool(feasible),
    }


def enumerate_candidates(step=0.5, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate a stated finite candidate grid."""
    levels = np.arange(MIN_COOLING, MAX_COOLING + step / 2, step)
    return [
        evaluate_formulation((early, late), energy_weight, energy_limit)
        for early in levels
        for late in levels
    ]


def select_best(records, key="f"):
    """Select the lowest-valued feasible record for the requested key."""
    return min(
        (record for record in records if record["feasible"]),
        key=lambda record: record[key] if key != "discomfort" else record["y"][key],
    )


def pareto_front(records):
    """Return nondominated feasible records for minimizing discomfort and energy."""
    feasible_records = [record for record in records if record["feasible"]]
    nondominated = []
    for candidate in feasible_records:
        candidate_d = candidate["y"]["discomfort"]
        candidate_e = candidate["y"]["energy"]
        dominated = any(
            other["y"]["discomfort"] <= candidate_d
            and other["y"]["energy"] <= candidate_e
            and (
                other["y"]["discomfort"] < candidate_d
                or other["y"]["energy"] < candidate_e
            )
            for other in feasible_records
        )
        if not dominated:
            nondominated.append(candidate)
    return sorted(nondominated, key=lambda record: record["y"]["energy"])

import sys
from types import SimpleNamespace

import matplotlib

def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt

def show_pareto_front(enumerate_candidates, pareto_front, *, max_energy=60.0):
    """Plot the feasible performance set and its Pareto front."""
    plt = _pyplot()
    records = enumerate_candidates(step=0.5, energy_weight=1.0, energy_limit=max_energy)
    feasible = [record for record in records if record["feasible"]]
    pareto = pareto_front(records)
    figure, axis = plt.subplots(figsize=(8.2, 5.2))
    axis.scatter(
        [record["y"]["energy"] for record in feasible],
        [record["y"]["discomfort"] for record in feasible],
        color="lightgray", edgecolor="white", s=62,
        label="Feasible candidate",
    )
    axis.plot(
        [record["y"]["energy"] for record in pareto],
        [record["y"]["discomfort"] for record in pareto],
        marker="o", color="teal", linewidth=2.2, markersize=6,
        label="Pareto front",
    )
    selected = (pareto[0], pareto[len(pareto) // 2], pareto[-1])
    for index, record in enumerate(selected):
        is_rightmost = index == len(selected) - 1
        axis.annotate(
            f"x=[{record['x'][0]:g}, {record['x'][1]:g}]^T",
            (record["y"]["energy"], record["y"]["discomfort"]),
            xytext=(-5 if is_rightmost else 7, 7),
            textcoords="offset points",
            ha="right" if is_rightmost else "left",
            fontsize=8,
        )
    axis.set(
        xlabel="Energy $E$",
        ylabel="Discomfort $D$",
        title="Feasible performance set and Pareto front",
    )
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    print(f"{len(feasible)} feasible grid candidates; {len(pareto)} nondominated candidates")
    return SimpleNamespace(feasible=feasible, pareto=pareto)

def show_formulation_lab(
    enumerate_candidates,
    select_best,
    pareto_front,
    *,
    max_energy=60.0,
):
    """Compare weighted-sum and epsilon-constraint selections interactively."""
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider

    plt.close("all")
    baseline = enumerate_candidates(step=0.5, energy_weight=1.0, energy_limit=max_energy)
    feasible = [record for record in baseline if record["feasible"]]
    pareto = pareto_front(baseline)

    def weighted_choice(energy_weight):
        records = enumerate_candidates(
            step=0.5, energy_weight=energy_weight, energy_limit=max_energy
        )
        return select_best(records)

    def epsilon_choice(energy_limit):
        records = enumerate_candidates(
            step=0.5, energy_weight=0.0, energy_limit=energy_limit
        )
        return select_best(records, key="discomfort")

    initial_weight = 1.0
    initial_limit = 45.0
    weighted = weighted_choice(initial_weight)
    epsilon = epsilon_choice(initial_limit)
    figure, axis = plt.subplots(figsize=(9.4, 7.2))
    figure.subplots_adjust(left=0.12, right=0.97, bottom=0.28, top=0.82)
    axis.scatter(
        [record["y"]["energy"] for record in feasible],
        [record["y"]["discomfort"] for record in feasible],
        color="lightgray", edgecolor="white", s=62,
        label="Feasible at E <= 60",
    )
    axis.plot(
        [record["y"]["energy"] for record in pareto],
        [record["y"]["discomfort"] for record in pareto],
        color="teal", linewidth=2, label="Pareto front",
    )
    weighted_marker = axis.scatter(
        weighted["y"]["energy"], weighted["y"]["discomfort"],
        marker="*", s=240, color="tab:purple", edgecolor="black",
        label="Weighted-sum selection", zorder=4,
    )
    epsilon_marker = axis.scatter(
        epsilon["y"]["energy"], epsilon["y"]["discomfort"],
        marker="D", s=95, color="tab:red", edgecolor="black",
        label="Epsilon-constraint selection", zorder=4,
    )
    limit_line = axis.axvline(
        initial_limit, color="tab:red", linestyle="--", linewidth=1.8,
        label="Energy limit $E_{\\max}$",
    )
    axis.set(
        xlabel="Energy $E$",
        ylabel="Discomfort $D$",
        title="Same candidates · different formulations",
        xlim=(0, 64),
    )
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    status_text = figure.text(
        0.5, 0.945, "", ha="center", va="top", fontsize=11, fontweight="bold"
    )
    detail_text = figure.text(0.5, 0.895, "", ha="center", va="top", fontsize=9)
    weight_axis = figure.add_axes([0.30, 0.145, 0.58, 0.03])
    limit_axis = figure.add_axes([0.30, 0.075, 0.58, 0.03])
    weight_slider = Slider(
        weight_axis,
        "Weighted sum · $\\lambda_E$",
        0.0,
        3.0,
        valinit=initial_weight,
        valstep=0.1,
        valfmt="%1.1f",
        color="tab:purple",
    )
    limit_slider = Slider(
        limit_axis,
        "$\\varepsilon$-constraint · $E_{\\max}$",
        10.0,
        max_energy,
        valinit=initial_limit,
        valstep=0.25,
        valfmt="%1.2f",
        color="tab:red",
    )
    state = {}

    def refresh(_=None):
        energy_weight = weight_slider.val
        energy_limit = limit_slider.val
        weighted = weighted_choice(energy_weight)
        epsilon = epsilon_choice(energy_limit)
        weighted_marker.set_offsets([[
            weighted["y"]["energy"], weighted["y"]["discomfort"]
        ]])
        epsilon_marker.set_offsets([[
            epsilon["y"]["energy"], epsilon["y"]["discomfort"]
        ]])
        limit_line.set_xdata([energy_limit, energy_limit])
        status_text.set_text(
            f"Weighted sum selects x=[{weighted['x'][0]:.1f}, {weighted['x'][1]:.1f}]^T  |  "
            f"epsilon constraint selects x=[{epsilon['x'][0]:.1f}, {epsilon['x'][1]:.1f}]^T"
        )
        detail_text.set_text(
            f"weighted: D={weighted['y']['discomfort']:.2f}, "
            f"E={weighted['y']['energy']:.2f}, lambda_E={energy_weight:.1f}  |  "
            f"epsilon: D={epsilon['y']['discomfort']:.2f}, "
            f"E={epsilon['y']['energy']:.2f} <= {energy_limit:.2f}"
        )
        state.clear()
        state.update(
            energy_weight=energy_weight,
            energy_limit=energy_limit,
            weighted=weighted,
            epsilon=epsilon,
        )
        figure.canvas.draw_idle()

    for slider in (weight_slider, limit_slider):
        slider.on_changed(refresh)
    figure._formulation_sliders = (weight_slider, limit_slider)
    refresh()
    plt.show()
    return SimpleNamespace(
        figure=figure,
        state=state,
        refresh=refresh,
        weight_slider=weight_slider,
        energy_limit_slider=limit_slider,
    )

pareto_summary = show_pareto_front(
    enumerate_candidates, pareto_front, max_energy=MAX_ENERGY
)


For example, candidate A with \((D,E)=(20,35)\) dominates candidate B with \((D,E)=(25,40)\). A is better in both minimized outputs. If B instead has \((D,E)=(15,40)\), neither dominates: B has less discomfort while A uses less energy.

When objectives conflict, improving one Pareto candidate requires accepting a worse value in another objective. A preference or additional requirement is needed to select one candidate.


### 3 · A weighted sum turns several objectives into one score

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad \sum_{i=1}^{I}w_i\widetilde f_i(y),\qquad w_i\ge0,\quad\sum_{i=1}^{I}w_i=1.$

Here, \(I\) is the number of objectives, \(w_i\) is the weight of objective \(i\), and \(\widetilde f_i\) is its normalized value. Normalization prevents units or numerical scale from silently controlling the comparison.

The classroom score \(D(u)+\lambda_EE(u)\) is an unnormalized weighted sum. The coefficient of \(D\) is fixed at 1, so \(\lambda_E\) is the relative energy penalty. Changing \(\lambda_E\) can change the selected candidate without changing the physical response of any fixed candidate.

### 4 · An \(\varepsilon\)-constraint turns one objective into a requirement

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad f_D(y)\qquad\text{subject to}\qquad f_E(y)\le\varepsilon.$

For the classroom, \(f_D(y)=D(u)\), \(f_E(y)=E(u)\), and \(\varepsilon=E_{\max}\). Discomfort remains the objective, while energy becomes a requirement. Tightening \(E_{\max}\) reduces the feasible set and can change the selected candidate.

The next figure applies both selection rules to the same feasible performance points. The purple control changes the weighted-sum preference. The red control changes the energy requirement.


In [ ]:
formulation_lab = show_formulation_lab(
    enumerate_candidates,
    select_best,
    pareto_front,
    max_energy=MAX_ENERGY,
)


### 5 · Separate physical change from evaluation change

For a fixed decision \(u\), the temperature path \(T\), discomfort \(D\), and energy \(E\) do not change when \(\lambda_E\) changes. Only the score and possibly the selected candidate change. The weight is a hyperparameter, not a physical control.

Changing \(E_{\max}\) also leaves the physical response of a fixed decision unchanged. It changes whether that candidate is eligible. Thus the two controls act on different parts of the formulation:

| Changed quantity | Direct effect | Possible selection effect |
|:---|:---|:---|
| Energy weight $\lambda_E$ | Changes the scalar comparison rule | A different feasible candidate may have the lowest score |
| Energy limit $E_{\max}$ | Changes the feasible set | A previously eligible candidate may be rejected |


### Takeaway

Classify the objective structure by asking how many values are minimized or maximized:

> **one scalar \(f\) → single-objective · vector \(\boldsymbol f\) → multi-objective · preference or requirement → one selected Pareto candidate**

Performance outputs describe a candidate. The objective supplies the comparison rule. A weighted sum changes the score, while an \(\varepsilon\)-constraint changes eligibility.

Part 5 keeps these formulation parts visible and changes the mathematical structure of the model and its treatment of uncertainty.
